# Figures 6 and 7: Forest and Grass at 37°N

This notebook produces just the two grey, four-panel figures

Replace `matrixlargeplantbase.npz` and `matrixsmallplant37.npz` with the results, rain from the main loop in the main file containing ss_vec (call ss), ssCAC, and CAC. You must relplace the atmospheric parameters in accordance from Cerasoli et al., which is found in the data sec in the same folder as this notebook, then use **Run All**. These `.npz` files contain results from the original model simulations; the notebook does not rerun the simulations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

length = 10000
dt = 0.01


def load_case(filename, required_keys):
    """Read one saved simulation without mixing it with the other case."""
    path = Path(filename)
    if not path.is_file():
        raise FileNotFoundError(
            f"Can't find {filename}. Put it next to this notebook and run again."
        )

    with np.load(path, allow_pickle=False) as data:
        missing = [key for key in required_keys if key not in data.files]
        if missing:
            raise KeyError(
                f"{filename} is missing {missing}. Available arrays: {data.files}"
            )
        return {key: np.asarray(data[key]) for key in required_keys}


def crossing_frequency_inputs(ss):
    """Use the same 50-day window and event counting as the original notebook."""
    jumpscare = np.zeros(len(ss))
    dt = 0.01

    for i in range(0, len(ss) - 1):
        if ss[i + 1] > ss[i]:
            jumpscare[i + 1] = 1
        else:
            jumpscare[i + 1] = 0

    steps_per_day = int(1 / dt)
    window = int(50 / dt)
    start_t = 50

    s_at_t = []
    freq_at_t = []
    last_start = len(jumpscare) - window

    for t in range(start_t, last_start, steps_per_day):
        s_at_t.append(ss[t])
        window_counts = np.sum(jumpscare[t : t + window])
        freq_at_t.append(window_counts / 50.0)

    # The final scatterplot expects X to be a column vector.
    X = np.asarray(s_at_t).reshape(-1, 1)
    y = np.asarray(freq_at_t)
    return X, y

## Forest

Load the forest simulation and calculate the event-frequency points used in the rightmost panel.

In [ ]:
forest = load_case("matrixlargeplantbase.npz", ("ss", "ssCAC", "CAC"))
ss = forest["ss"]
ssCAC = forest["ssCAC"]
CAC = forest["CAC"]

days = np.linspace(0, length, len(ss))
X, y = crossing_frequency_inputs(ss)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 18,
    "axes.labelsize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.linewidth": 2.0
})

fig, axes = plt.subplots(1, 4, figsize=(24, 4), dpi=300, constrained_layout=True,)
fig.suptitle("Forest", fontsize=22)
t = np.asarray(days) if "days" in globals() else np.arange(len(ss))

# --- Soil moisture time series ---
axes[0].plot(t, ss, color="0.4", linewidth=1.0)

axes[0].axhline(np.mean(ss), color="0.7", linestyle="--", linewidth=1.2)

axes[0].set_xlabel("Time (days)" if "days" in globals() else "Index")
axes[0].set_ylabel(r"$s$")
axes[0].tick_params(direction="out")
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# --- CAPE histogram ---
axes[1].hist(
    CAC[CAC != 0],
    bins=25,
    density=True,
    facecolor="0.7",
    edgecolor="black",
    linewidth=1.5
)

axes[1].set_xlabel("CAPE at crossing time")
axes[1].set_ylabel("Density")
axes[1].tick_params(direction="out")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

# --- Soil moisture histogram ---
axes[2].hist(
    ssCAC,
    bins=25,
    density=True,
    facecolor="0.7",
    edgecolor="black",
    linewidth=1.5
)

axes[2].set_xlabel(r"$s$ at Crossing Time")
axes[2].set_ylabel("Density")
axes[2].tick_params(direction="out")
axes[2].spines["top"].set_visible(False)
axes[2].spines["right"].set_visible(False)

# --- Scatter + linear fit ---
axes[3].scatter(X, y, s=10, alpha=0.15, color="0.4")

x = X[:, 0]
mask = np.isfinite(x) & np.isfinite(y)
m, b = np.polyfit(x[mask], y[mask], 1)

xx = np.linspace(x[mask].min(), x[mask].max(), 200)
axes[3].plot(xx, m * xx + b, color="0.4", linewidth=2)

axes[3].set_xlabel(r"$s_i$")
axes[3].set_ylabel(r"$\overline{\lambda}$")
axes[3].tick_params(direction="out")
axes[3].spines["top"].set_visible(False)
axes[3].spines["right"].set_visible(False)

plt.savefig("samplelargebase", bbox_inches="tight")
plt.show()


## Grass at 37°N

Switch to the grass simulation, then recalculate the same event-frequency inputs for this case. This keeps the grass panels independent of whatever was plotted for the forest.

In [ ]:
grass = load_case("matrixsmallplant37.npz", ("ss", "CAC"))
ss = grass["ss"]
CAC = grass["CAC"]

days = np.linspace(0, length, len(ss))
X, y = crossing_frequency_inputs(ss)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 18,
    "axes.labelsize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.linewidth": 2.0
})

fig, axes = plt.subplots(1, 4, figsize=(24, 4), dpi=300, constrained_layout=True)
fig.suptitle("Grass at 37°N", fontsize=22)

t = np.asarray(days) if "days" in globals() else np.arange(len(ss))

# --- Soil moisture time series ---
axes[0].plot(t, ss, color="0.4", linewidth=1.0)
axes[0].axhline(np.mean(ss), color="0.7", linestyle="--", linewidth=1.2)
axes[0].set_xlabel("Time (days)" if "days" in globals() else "Index")
axes[0].set_ylabel(r"$s$")
axes[0].tick_params(direction="out")
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# --- CAPE histogram or blank panel ---
cape_data = CAC[CAC != 0] if "CAC" in globals() else np.array([])

if len(cape_data) > 0:
    axes[1].hist(
        cape_data,
        bins=25,
        density=True,
        facecolor="0.7",
        edgecolor="black",
        linewidth=1.5
    )
    axes[1].set_xlabel("CAPE at crossing time")
    axes[1].set_ylabel("Density")
else:
    axes[1].text(
        0.5, 0.5, "No crossing\nevents",
        ha="center", va="center", fontsize=18,
        transform=axes[1].transAxes
    )
    axes[1].set_xlabel("CAPE at crossing time")
    axes[1].set_ylabel("Density")
    axes[1].set_xticks([])
    axes[1].set_yticks([])

axes[1].tick_params(direction="out")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

# --- Soil moisture histogram or blank panel ---
ss_cross_data = ss #ssCAC if "ssCAC" in globals() else np.array([])

if len(ss_cross_data) > 0:
    axes[2].hist(
        ss_cross_data,
        bins=25,
        density=True,
        facecolor="0.7",
        edgecolor="black",
        linewidth=1.5
    )
    axes[2].set_xlabel(r"$s$")
    axes[2].set_ylabel("Density")
else:
    axes[2].text(
        0.5, 0.5, "No crossing\nevents",
        ha="center", va="center", fontsize=18,
        transform=axes[2].transAxes
    )
    axes[2].set_xlabel(r"$s$ at crossing time")
    axes[2].set_ylabel("Density")
    axes[2].set_xticks([])
    axes[2].set_yticks([])

axes[2].tick_params(direction="out")
axes[2].spines["top"].set_visible(False)
axes[2].spines["right"].set_visible(False)

# --- Scatter + linear fit ---
axes[3].scatter(X, y, s=10, alpha=0.15, color="0.4")

x = X[:, 0]
mask = np.isfinite(x) & np.isfinite(y)

if np.sum(mask) >= 2:
    m, b = np.polyfit(x[mask], y[mask], 1)
    xx = np.linspace(x[mask].min(), x[mask].max(), 200)
    axes[3].plot(xx, m * xx + b, color="0.4", linewidth=2)

axes[3].set_xlabel(r"$s_i$")
axes[3].set_ylabel(r"$\overline{\lambda}$")
axes[3].tick_params(direction="out")
axes[3].spines["top"].set_visible(False)
axes[3].spines["right"].set_visible(False)

plt.savefig("samplesmallbase", bbox_inches="tight")
plt.show()